# PINN vs HNN vs Classical Solver — Comprehensive Comparison

This notebook compares three approaches to modeling pendulum dynamics:
1. **Standard PINN** — learns the trajectory directly from the ODE
2. **Hamiltonian Neural Network (HNN)** — learns the energy function H(q,p) and derives dynamics via autograd
3. **Classical solver** (scipy RK45) — numerical ground truth

We evaluate:
- Energy conservation over long time horizons
- Trajectory accuracy vs scipy
- Inverse problem: recovering unknown physical parameters from noisy data

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Physical parameters
g = 9.81
L = 1.0
m = 1.0
theta_0 = np.pi / 4  # 45 degrees
omega_0 = 0.0
t_max = 10.0

print("Setup complete.")

## 1. Model Definitions

We define all three models inline so this notebook is self-contained.

In [ ]:
# --- Standard PINN: maps t -> (theta, omega) ---
class PINNPendulum(nn.Module):
    def __init__(self, hidden_size=64, num_hidden_layers=3):
        super().__init__()
        layers = [nn.Linear(1, hidden_size), nn.Tanh()]
        for _ in range(num_hidden_layers - 1):
            layers += [nn.Linear(hidden_size, hidden_size), nn.Tanh()]
        layers.append(nn.Linear(hidden_size, 2))
        self.network = nn.Sequential(*layers)

    def forward(self, t):
        return self.network(t)


# --- Hamiltonian Neural Network: maps (q, p) -> H ---
class HamiltonianNet(nn.Module):
    def __init__(self, hidden_size=64, num_hidden_layers=3):
        super().__init__()
        layers = [nn.Linear(2, hidden_size), nn.Tanh()]
        for _ in range(num_hidden_layers - 1):
            layers += [nn.Linear(hidden_size, hidden_size), nn.Tanh()]
        layers.append(nn.Linear(hidden_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, q, p):
        return self.network(torch.cat([q, p], dim=1))

    def time_derivatives(self, q, p):
        H = self.forward(q, p)
        dH_dq = torch.autograd.grad(H, q, torch.ones_like(H), create_graph=True)[0]
        dH_dp = torch.autograd.grad(H, p, torch.ones_like(H), create_graph=True)[0]
        return dH_dp, -dH_dq  # dq/dt, dp/dt


print("Models defined.")

## 2. Training

### 2a. Train Standard PINN

In [ ]:
def train_pinn(theta_0, omega_0, t_max, g, L, epochs=5000, n_col=500, ic_weight=20.0):
    model = PINNPendulum()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=500, factor=0.5, min_lr=1e-6)
    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        t_col = torch.rand(n_col, 1) * t_max
        t_col.requires_grad_(True)
        out = model(t_col)
        theta, omega = out[:, 0:1], out[:, 1:2]

        dtheta = torch.autograd.grad(theta, t_col, torch.ones_like(theta), create_graph=True)[0]
        domega = torch.autograd.grad(omega, t_col, torch.ones_like(omega), create_graph=True)[0]

        loss_phys = torch.mean((dtheta - omega)**2) + torch.mean((domega + (g/L)*torch.sin(theta))**2)

        out0 = model(torch.zeros(1, 1))
        loss_ic = (out0[0, 0] - theta_0)**2 + (out0[0, 1] - omega_0)**2

        loss = loss_phys + ic_weight * loss_ic
        loss.backward()
        optimizer.step()
        scheduler.step(loss.item())
        losses.append(loss.item())

        if (epoch + 1) % 1000 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {loss.item():.6f}")

    return model, losses

print("Training Standard PINN...")
pinn_model, pinn_losses = train_pinn(theta_0, omega_0, t_max, g, L)
print(f"Done. Final loss: {pinn_losses[-1]:.6f}")

### 2b. Train Hamiltonian Neural Network

In [ ]:
# Generate training data for HNN from true dynamics
t_data = np.linspace(0, t_max, 1000)
sol = solve_ivp(lambda t, y: [y[1], -(g/L)*np.sin(y[0])],
                (0, t_max), [theta_0, omega_0],
                t_eval=t_data, method='RK45', rtol=1e-10, atol=1e-12)

q_data = sol.y[0]
p_data = m * L**2 * sol.y[1]
dqdt_data = p_data / (m * L**2)
dpdt_data = -m * g * L * np.sin(q_data)

def train_hnn(q_data, p_data, dqdt_data, dpdt_data, epochs=5000, batch_size=256):
    model = HamiltonianNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=500, factor=0.5, min_lr=1e-6)

    q_t = torch.tensor(q_data, dtype=torch.float32).unsqueeze(1)
    p_t = torch.tensor(p_data, dtype=torch.float32).unsqueeze(1)
    dq_t = torch.tensor(dqdt_data, dtype=torch.float32).unsqueeze(1)
    dp_t = torch.tensor(dpdt_data, dtype=torch.float32).unsqueeze(1)
    n = len(q_data)
    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        idx = torch.randint(0, n, (batch_size,))
        q_b = q_t[idx].requires_grad_(True)
        p_b = p_t[idx].requires_grad_(True)

        dq_pred, dp_pred = model.time_derivatives(q_b, p_b)
        loss = torch.mean((dq_pred - dq_t[idx])**2) + torch.mean((dp_pred - dp_t[idx])**2)

        loss.backward()
        optimizer.step()
        scheduler.step(loss.item())
        losses.append(loss.item())

        if (epoch + 1) % 1000 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {loss.item():.8f}")

    return model, losses

print("Training HNN...")
hnn_model, hnn_losses = train_hnn(q_data, p_data, dqdt_data, dpdt_data)
print(f"Done. Final loss: {hnn_losses[-1]:.8f}")

## 3. Comparison 1: PINN vs HNN Energy Conservation

This is the key comparison. The HNN conserves energy by construction (symplectic structure), while the standard PINN only conserves energy as well as its training loss allows.

In [ ]:
t_eval = np.linspace(0, t_max, 1000)

# --- Ground truth ---
sol_true = solve_ivp(lambda t, y: [y[1], -(g/L)*np.sin(y[0])],
                     (0, t_max), [theta_0, omega_0],
                     t_eval=t_eval, method='RK45', rtol=1e-10, atol=1e-12)
theta_true, omega_true = sol_true.y[0], sol_true.y[1]
q_true, p_true = theta_true, m * L**2 * omega_true

# --- PINN prediction ---
pinn_model.eval()
with torch.no_grad():
    pinn_out = pinn_model(torch.tensor(t_eval, dtype=torch.float32).unsqueeze(1)).numpy()
theta_pinn, omega_pinn = pinn_out[:, 0], pinn_out[:, 1]
p_pinn = m * L**2 * omega_pinn

# --- HNN prediction (integrate learned dynamics) ---
hnn_model.eval()
def hnn_rhs(t, state):
    q, p = state
    q_t = torch.tensor([[q]], dtype=torch.float32, requires_grad=True)
    p_t = torch.tensor([[p]], dtype=torch.float32, requires_grad=True)
    with torch.enable_grad():
        dqdt, dpdt = hnn_model.time_derivatives(q_t, p_t)
    return [dqdt.item(), dpdt.item()]

sol_hnn = solve_ivp(hnn_rhs, (0, t_max), [theta_0, m*L**2*omega_0],
                    t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-10)
q_hnn, p_hnn = sol_hnn.y[0], sol_hnn.y[1]
omega_hnn = p_hnn / (m * L**2)

# --- Compute energies ---
def energy(q, p):
    return p**2 / (2*m*L**2) - m*g*L*np.cos(q)

E_true = energy(q_true, p_true)
E_pinn = energy(theta_pinn, p_pinn)
E_hnn = energy(q_hnn, p_hnn)
E0 = E_true[0]

dE_true = np.abs((E_true - E0) / (np.abs(E0) + 1e-16))
dE_pinn = np.abs((E_pinn - E0) / (np.abs(E0) + 1e-16))
dE_hnn = np.abs((E_hnn - E0) / (np.abs(E0) + 1e-16))

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Energy over time
ax1 = axes[0]
ax1.plot(t_eval, E_true, 'b-', lw=2, label='RK45 (truth)')
ax1.plot(t_eval, E_hnn, 'g-', lw=2, label='HNN')
ax1.plot(t_eval, E_pinn, 'r-', lw=2, label='Standard PINN')
ax1.axhline(y=E0, color='gray', ls=':', alpha=0.5)
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Total Energy')
ax1.set_title('Energy Conservation'); ax1.legend(); ax1.grid(True, alpha=0.3)

# Relative energy error
ax2 = axes[1]
ax2.semilogy(t_eval, dE_true + 1e-16, 'b-', lw=2, label='RK45')
ax2.semilogy(t_eval, dE_hnn + 1e-16, 'g-', lw=2, label='HNN')
ax2.semilogy(t_eval, dE_pinn + 1e-16, 'r-', lw=2, label='PINN')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('|dE/E_0|')
ax2.set_title('Relative Energy Error'); ax2.legend(); ax2.grid(True, alpha=0.3)

# Phase portrait
ax3 = axes[2]
ax3.plot(np.degrees(q_true), omega_true, 'b-', lw=2, label='RK45', alpha=0.7)
ax3.plot(np.degrees(q_hnn), omega_hnn, 'g--', lw=2, label='HNN')
ax3.plot(np.degrees(theta_pinn), omega_pinn, 'r--', lw=2, label='PINN')
ax3.set_xlabel('Angle (deg)'); ax3.set_ylabel('Angular Velocity (rad/s)')
ax3.set_title('Phase Portrait'); ax3.legend(); ax3.grid(True, alpha=0.3)

plt.suptitle('PINN vs HNN — Energy Conservation Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nMax |dE/E_0|:")
print(f"  RK45 (scipy):    {np.max(dE_true):.2e}")
print(f"  HNN:             {np.max(dE_hnn):.2e}")
print(f"  Standard PINN:   {np.max(dE_pinn):.2e}")
print(f"\nHNN energy error is {np.max(dE_pinn)/(np.max(dE_hnn)+1e-16):.1f}x better than PINN")

## 4. Comparison 2: PINN vs scipy Accuracy Over Long Time Horizons

We extend the simulation to 30 seconds (about 5 pendulum periods) to see how the PINN's error accumulates compared to the classical solver. PINNs are trained on a fixed time domain, so extrapolation beyond that domain degrades rapidly — but even within the domain, accuracy can drift at late times.

In [ ]:
# Train a PINN on a longer domain for fair comparison
t_max_long = 30.0

print("Training PINN on extended domain (30s)...")
pinn_long, _ = train_pinn(theta_0, omega_0, t_max_long, g, L, epochs=8000, n_col=800)

t_long = np.linspace(0, t_max_long, 3000)

# Ground truth (high-accuracy scipy)
sol_long = solve_ivp(lambda t, y: [y[1], -(g/L)*np.sin(y[0])],
                     (0, t_max_long), [theta_0, omega_0],
                     t_eval=t_long, method='RK45', rtol=1e-12, atol=1e-14)
theta_true_long = sol_long.y[0]

# PINN prediction
pinn_long.eval()
with torch.no_grad():
    pinn_out_long = pinn_long(torch.tensor(t_long, dtype=torch.float32).unsqueeze(1)).numpy()
theta_pinn_long = pinn_out_long[:, 0]

# Error analysis
abs_error = np.abs(theta_pinn_long - theta_true_long)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(t_long, np.degrees(theta_true_long), 'b-', lw=2, label='scipy RK45', alpha=0.7)
ax1.plot(t_long, np.degrees(theta_pinn_long), 'r--', lw=2, label='PINN')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Angle (degrees)')
ax1.set_title('PINN vs scipy — 30s Horizon')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.semilogy(t_long, abs_error + 1e-16, 'g-', lw=2)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('|theta_PINN - theta_true| (rad)')
ax2.set_title('PINN Absolute Error Over Time')
ax2.grid(True, alpha=0.3)

# Mark the original training domain
ax2.axvline(x=t_max, color='orange', ls='--', alpha=0.7, label=f'Original t_max={t_max}s')
ax2.legend()

plt.suptitle('Long-Horizon Accuracy: PINN vs Classical Solver', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nMax error in [0, {t_max}s]:    {np.max(abs_error[:1000]):.6f} rad")
print(f"Max error in [{t_max}, {t_max_long}s]:  {np.max(abs_error[1000:]):.6f} rad")
print(f"Error growth factor:          {np.max(abs_error[1000:])/(np.max(abs_error[:1000])+1e-16):.1f}x")

## 5. Comparison 3: Inverse Problem — Recovering Physical Parameters

We generate noisy observations of a pendulum with known g=9.81, L=1.0, then use an inverse PINN starting from wrong initial guesses (g=5.0, L=2.0) to recover the true parameters.

In [ ]:
# Generate noisy data
g_true, L_true = 9.81, 1.0
n_obs = 50
noise_std = 0.05

t_obs = np.sort(np.random.uniform(0.1, t_max, n_obs))
sol_obs = solve_ivp(lambda t, y: [y[1], -(g_true/L_true)*np.sin(y[0])],
                    (0, t_max), [theta_0, omega_0],
                    t_eval=t_obs, method='RK45', rtol=1e-10, atol=1e-12)
theta_obs = sol_obs.y[0] + np.random.normal(0, noise_std, n_obs)
omega_obs = sol_obs.y[1] + np.random.normal(0, noise_std, n_obs)

# Train inverse PINN
inv_model = PINNPendulum()
g_param = torch.tensor(5.0, dtype=torch.float32, requires_grad=True)
L_param = torch.tensor(2.0, dtype=torch.float32, requires_grad=True)

optimizer = torch.optim.Adam([
    {'params': inv_model.parameters(), 'lr': 1e-3},
    {'params': [g_param, L_param], 'lr': 1e-2}
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=500, factor=0.5, min_lr=1e-6)

t_obs_t = torch.tensor(t_obs, dtype=torch.float32).unsqueeze(1)
theta_obs_t = torch.tensor(theta_obs, dtype=torch.float32)
omega_obs_t = torch.tensor(omega_obs, dtype=torch.float32)

g_hist, L_hist = [], []
print("Training Inverse PINN (g_init=5.0, L_init=2.0)...")

for epoch in range(8000):
    optimizer.zero_grad()
    
    # Physics loss
    t_col = torch.rand(500, 1) * t_max
    t_col.requires_grad_(True)
    out = inv_model(t_col)
    th, om = out[:, 0:1], out[:, 1:2]
    dth = torch.autograd.grad(th, t_col, torch.ones_like(th), create_graph=True)[0]
    dom = torch.autograd.grad(om, t_col, torch.ones_like(om), create_graph=True)[0]
    loss_phys = torch.mean((dth - om)**2) + torch.mean((dom + (g_param/L_param)*torch.sin(th))**2)
    
    # Data loss
    out_obs = inv_model(t_obs_t)
    loss_data = torch.mean((out_obs[:, 0] - theta_obs_t)**2) + torch.mean((out_obs[:, 1] - omega_obs_t)**2)
    
    # IC loss
    out0 = inv_model(torch.zeros(1, 1))
    loss_ic = (out0[0, 0] - theta_0)**2 + (out0[0, 1] - omega_0)**2
    
    loss = loss_phys + 10.0 * loss_data + 20.0 * loss_ic
    loss.backward()
    optimizer.step()
    scheduler.step(loss.item())
    
    with torch.no_grad():
        g_param.clamp_(min=0.1)
        L_param.clamp_(min=0.01)
    
    g_hist.append(g_param.item())
    L_hist.append(L_param.item())
    
    if (epoch + 1) % 2000 == 0:
        print(f"  Epoch {epoch+1} | g={g_param.item():.4f} | L={L_param.item():.4f}")

print(f"\nFinal: g={g_param.item():.4f} (true: 9.81), L={L_param.item():.4f} (true: 1.00)")
print(f"Error: g={abs(g_param.item()-9.81)/9.81*100:.2f}%, L={abs(L_param.item()-1.0)*100:.2f}%")

In [ ]:
# Visualize inverse problem convergence
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax1 = axes[0]
ax1.plot(g_hist, 'r-', lw=1.5, label='Inferred g')
ax1.axhline(y=9.81, color='blue', ls='--', lw=2, label='True g = 9.81')
ax1.fill_between(range(len(g_hist)), 9.81*0.99, 9.81*1.01, color='blue', alpha=0.1, label='1% band')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('g (m/s^2)')
ax1.set_title(f'Gravity Convergence (final: {g_hist[-1]:.4f})')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(L_hist, 'r-', lw=1.5, label='Inferred L')
ax2.axhline(y=1.0, color='blue', ls='--', lw=2, label='True L = 1.0')
ax2.fill_between(range(len(L_hist)), 0.99, 1.01, color='blue', alpha=0.1, label='1% band')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('L (m)')
ax2.set_title(f'Length Convergence (final: {L_hist[-1]:.4f})')
ax2.legend(); ax2.grid(True, alpha=0.3)

# Trajectory reconstruction
ax3 = axes[2]
t_plot = np.linspace(0, t_max, 1000)
inv_model.eval()
with torch.no_grad():
    inv_out = inv_model(torch.tensor(t_plot, dtype=torch.float32).unsqueeze(1)).numpy()

sol_true_inv = solve_ivp(lambda t, y: [y[1], -(g_true/L_true)*np.sin(y[0])],
                         (0, t_max), [theta_0, omega_0],
                         t_eval=t_plot, method='RK45', rtol=1e-10, atol=1e-12)

ax3.plot(t_plot, np.degrees(sol_true_inv.y[0]), 'b-', lw=2, label='Ground truth', alpha=0.7)
ax3.scatter(t_obs, np.degrees(theta_obs), c='gray', s=20, alpha=0.5, label='Noisy data')
ax3.plot(t_plot, np.degrees(inv_out[:, 0]), 'r--', lw=2, label='PINN reconstruction')
ax3.set_xlabel('Time (s)'); ax3.set_ylabel('Angle (degrees)')
ax3.set_title('Trajectory Reconstruction')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

plt.suptitle('Inverse Problem: Inferring g and L from Noisy Observations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| **Standard PINN** | No training data needed, just the ODE. Continuous solution. | Energy conservation depends on training quality. Accuracy degrades at long horizons. |
| **HNN** | Energy conservation by construction (symplectic structure). Robust over long times. | Needs training data (state-derivative pairs). Cannot directly solve forward from ICs alone. |
| **Inverse PINN** | Simultaneously infers unknown physics and reconstructs trajectories. Works with noisy data. | Harder optimization (two parameter groups). Sensitive to loss weighting. |
| **scipy RK45** | Machine-precision accuracy with adaptive stepping. | Discrete solution only. No parameter inference. No generalization. |

**Key takeaway**: PINNs and HNNs are complementary. The PINN excels when you have the equation but no data. The HNN excels when you need structural guarantees (energy conservation). The inverse PINN bridges the gap between data-driven and physics-based approaches — essential for real-world applications where physical parameters are uncertain.